# Phase 3: Targeted Voice Artifact Synthesis (GGUF)
This notebook uses the highly-optimized Python and `llama_cpp` integration to synthesize voice artifacts using the Svara-TTS `Q3_K_S.gguf` model.

In [ ]:
!pip install -q llama-cpp-python onnxruntime soundfile huggingface_hub numpy

from huggingface_hub import hf_hub_download
import os

print("Downloading SNAC ONNX Decoder...")
SNAC_DECODER = hf_hub_download(repo_id="onnx-community/snac_24khz-ONNX", filename="onnx/decoder_model.onnx")

print("Downloading Svara TTS Q3_K_S GGUF...")
MODEL_PATH = hf_hub_download(repo_id="mradermacher/svara-tts-v1-GGUF", filename="svara-tts-v1.Q3_K_S.gguf")

print(f"Decoder path: {SNAC_DECODER}")
print(f"Model path: {MODEL_PATH}")


In [ ]:
import json, os, time, re
from pathlib import Path
import numpy as np
import onnxruntime as ort
import soundfile as sf
from llama_cpp import Llama

CONFIG = {
    "docling_input_dataset": "/kaggle/input/datasets/akashgoundi/curriculum-json/idp_curriculum_generation",
    "artifact_input_dataset": "/kaggle/input/datasets/akashgoundi/idp-ai-1st-output/idp_curriculum_generation",
    "output_root": "/kaggle/working/idp_curriculum_generation"
}

OUTPUT_ROOT = Path(CONFIG["output_root"])
DOCLING_ROOT = Path(CONFIG.get("docling_input_dataset", CONFIG["output_root"]))
ARTIFACT_ROOT = Path(CONFIG.get("artifact_input_dataset", CONFIG["output_root"]))

PACK_ROOT = ARTIFACT_ROOT / "generated_pack"
STRUCTURED_ROOT = DOCLING_ROOT / "structured_chapters"

VOICE_ROOT = OUTPUT_ROOT / "generated_pack_with_voice" / "voice"
for d in ["section_summaries", "concepts", "glossary", "faqs", "doubts", "flashcards"]:
    (VOICE_ROOT / d).mkdir(parents=True, exist_ok=True)

print(f"Reading structured PDFs from: {STRUCTURED_ROOT}")
print(f"Reading artifacts from: {PACK_ROOT}")
print(f"Writing voice outputs to: {VOICE_ROOT}")

SAMPLE_RATE = 24000
PAUSE_SAMPLES = int(1.5 * SAMPLE_RATE)

def snac_codes(tokens, input_len):
    generated = [int(x) for x in tokens[input_len:]]
    start = generated.index(128257) + 1 if 128257 in generated else 0
    audio = []
    band_pos = 0
    for token in generated[start:]:
        if token == 128258: break
        if token < 128266 or token >= 156938: continue
        band = band_pos % 7
        code = token - 128266 - band * 4096
        if 0 <= code < 4096:
            audio.append((band, code))
            band_pos += 1
    frames = len(audio) // 7
    level0, level1, level2 = [], [], []
    for i in range(frames):
        codes = [c for _, c in audio[i * 7 : i * 7 + 7]]
        level0.append(codes[0])
        level1.extend([codes[1], codes[4]])
        level2.extend([codes[2], codes[3], codes[5], codes[6]])
    return {
        "frames": frames,
        "level0": np.asarray([level0], dtype=np.int64),
        "level1": np.asarray([level1], dtype=np.int64),
        "level2": np.asarray([level2], dtype=np.int64),
    }

def decode_snac(session, codes):
    if codes["frames"] < 1: return np.array([], dtype=np.float32)
    out = session.run(None, {
        "audio_codes.0": codes["level0"],
        "audio_codes.1": codes["level1"],
        "audio_codes.2": codes["level2"],
    })
    return np.asarray(out[0]).reshape(-1)

def chunk_text(text):
    if not text: return []
    # Strict sentence-by-sentence chunking to prevent the TTS model from early stopping at periods
    sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', text.replace('\n', ' ')) if s.strip()]
    if not sentences: sentences = [text]
    return sentences

report = {"total_files": 0, "start_time": time.time(), "errors": [], "categories": {}}
manifest = {"chapter_id": "unknown", "summary": "", "objectives": "", "full_narration": "", "sections": [], "concepts": [], "glossary": [], "faqs": [], "doubts": [], "flashcards": []}

print("Loading models...")
start_load = time.time()
llm = Llama(model_path=MODEL_PATH, n_ctx=32768, n_threads=8, n_gpu_layers=0, logits_all=False, verbose=False)
decoder = ort.InferenceSession(SNAC_DECODER, providers=["CPUExecutionProvider"])
print(f"Models loaded in {time.time() - start_load:.1f}s")

def synthesize_wav(text_chunks, output_path, category, insert_pause=False):
    if output_path.exists(): 
        print(f"  [CACHE HIT] {output_path.name}")
        return str(output_path.relative_to(VOICE_ROOT))
    
    print(f"\nSynthesizing [{category}]: {output_path.name} ({len(text_chunks)} chunks)")
    synth_start = time.time()
    combined_samples = []
    for i, chunk in enumerate(text_chunks):
        if not chunk: continue
        try:
            chunk_start = time.time()
            body = llm.tokenize(f"English (Indian) (Female): {chunk}".encode("utf-8"), add_bos=False, special=True)
            ids = [128259, llm.token_bos()] + list(body) + [128009, 128260]
            
            generated = []
            for token in llm.generate(ids, temp=0.6, top_k=40, top_p=0.9, repeat_penalty=1.0, reset=True):
                generated.append(int(token))
                if int(token) == 128258 or len(generated) >= 4000: break
                    
            codes = snac_codes(ids + generated, len(ids))
            audio = decode_snac(decoder, codes)
            if len(audio) > 0: 
                combined_samples.append(audio)
            
            pause_msg = ""
            if insert_pause and i < len(text_chunks) - 1:
                combined_samples.append(np.zeros(PAUSE_SAMPLES, dtype=np.float32))
                pause_msg = " [+1.5s pause]"
                
            print(f"  -> Chunk {i+1}/{len(text_chunks)} ({len(audio)} samples) in {time.time() - chunk_start:.1f}s{pause_msg}")
            
        except Exception as e:
            err_msg = f"Error chunk '{chunk[:30]}...': {e}"
            print(f"  -> [FAILED] {err_msg}")
            report["errors"].append(err_msg)
            
    if combined_samples:
        final_audio = np.concatenate(combined_samples)
        sf.write(output_path, final_audio, SAMPLE_RATE)
        report["total_files"] += 1
        report["categories"][category] = report["categories"].get(category, 0) + 1
        print(f"  [SUCCESS] Saved {len(final_audio)} total samples in {time.time() - synth_start:.1f}s")
        return str(output_path.relative_to(VOICE_ROOT))
    
    print(f"  [FATAL] Failed to generate {output_path.name}")
    report["errors"].append(f"Failed to generate: {output_path.name}")
    return None

def clean_slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:50]

def read_json(fname):
    p = PACK_ROOT / fname
    return json.loads(p.read_text("utf-8")) if p.exists() else []

# 1. SECTION SUMMARIES
all_summaries = []
for s in read_json("summary.json"):
    if s.get("chapter_id"): manifest["chapter_id"] = s["chapter_id"]
    txt = s.get("payload", {}).get("summary_short", "")
    if txt:
        all_summaries.append(txt)
        p = synthesize_wav(chunk_text(txt), VOICE_ROOT / "section_summaries" / f"{s.get('section_id')}.wav", "section_summaries")
        if p: manifest["sections"].append(p)

if all_summaries:
    p = synthesize_wav(chunk_text(" ".join(all_summaries)), VOICE_ROOT / "chapter_summary.wav", "section_summaries")
    if p: manifest["summary"] = p

# 2. OBJECTIVES
obj_texts = []
for o in read_json("learning_objectives.json"):
    for i in o.get("payload", {}).get("learning_objectives", []):
        obj_texts.append(i.get("objective", ""))
if obj_texts:
    p = synthesize_wav(chunk_text(" ".join(obj_texts)), VOICE_ROOT / "learning_objectives.wav", "section_summaries")
    if p: manifest["objectives"] = p

# 3. CONCEPTS
for s in read_json("concepts.json"):
    for c in s.get("payload", {}).get("concepts", []):
        if c.get("name") and c.get("definition"):
            txt = f"{c['name']}. {c['definition']}"
            p = synthesize_wav(chunk_text(txt), VOICE_ROOT / "concepts" / f"{clean_slug(c['name'])}.wav", "concepts")
            if p: manifest["concepts"].append(p)

# 4. GLOSSARY
for s in read_json("glossary.json"):
    for g in s.get("payload", {}).get("glossary", []):
        if g.get("term"):
            txt = f"{g['term']}. {g.get('definition', '')}"
            p = synthesize_wav(chunk_text(txt), VOICE_ROOT / "glossary" / f"{clean_slug(g['term'])}.wav", "glossary")
            if p: manifest["glossary"].append(p)

# 5. FAQS
idx = 1
for s in read_json("faqs.json"):
    for f in s.get("payload", {}).get("items", []):
        txt = f"{f.get('question', '')}. {f.get('answer', '')}"
        p = synthesize_wav(chunk_text(txt), VOICE_ROOT / "faqs" / f"faq_{idx:03d}.wav", "faqs")
        if p: manifest["faqs"].append(p)
        idx += 1

# 6. DOUBTS
idx = 1
for s in read_json("common_doubts.json"):
    for d in s.get("payload", {}).get("items", []):
        txt = f"{d.get('doubt', '')}. {d.get('explanation', '')}"
        p = synthesize_wav(chunk_text(txt), VOICE_ROOT / "doubts" / f"doubt_{idx:03d}.wav", "doubts")
        if p: manifest["doubts"].append(p)
        idx += 1

# 7. FLASHCARDS
idx = 1
for s in read_json("flashcards.json"):
    for f in s.get("payload", {}).get("items", []):
        if f.get("front") and f.get("back"):
            parts = chunk_text(f["front"]) + chunk_text(f["back"])
            p = synthesize_wav(parts, VOICE_ROOT / "flashcards" / f"flashcard_{idx:03d}.wav", "flashcards", insert_pause=True)
            if p: manifest["flashcards"].append(p)
            idx += 1

# 8. DETAILED EXPLANATION (Replacing Full Narration)
full_text = ""
for s in read_json("detailed_explanation.json"):
    txt = s.get("payload", {}).get("explanation", "")
    if txt:
        full_text += txt + " "
if full_text.strip():
    p = synthesize_wav(chunk_text(full_text, 30), VOICE_ROOT / "detailed_explanation.wav", "full_narration")
    if p: manifest["full_narration"] = p

# Wrap up
(VOICE_ROOT / "voice_manifest.json").write_text(json.dumps(manifest, indent=2))
report["total_time_seconds"] = round(time.time() - report["start_time"])
(VOICE_ROOT / "voice_generation_report.json").write_text(json.dumps(report, indent=2))
print("\n\n==============================")
print(f"Voice generation complete in {report['total_time_seconds']}s")
print(f"Total files generated: {report['total_files']}")
print(f"Errors encountered: {len(report['errors'])}")
print("==============================")


In [ ]:
import zipfile
OUTPUT_ZIP_DIR = OUTPUT_ROOT / "generated_pack_with_voice"
OUTPUT_ZIP_DIR.mkdir(parents=True, exist_ok=True)
zip_path = OUTPUT_ROOT / "generated_pack_with_voice.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    # Add all JSON artifacts from the read-only Input PACK_ROOT into the zip under 'generated_pack/'
    for file_path in PACK_ROOT.rglob("*.json"):
        archive.write(file_path, arcname=f"generated_pack/{file_path.relative_to(PACK_ROOT)}")

    # Add all Voice artifacts into the zip under 'generated_pack/voice/'
    for file_path in VOICE_ROOT.rglob("*.wav"):
        archive.write(file_path, arcname=f"generated_pack/voice/{file_path.relative_to(VOICE_ROOT)}")
    
    voice_manifest = VOICE_ROOT / "voice_manifest.json"
    if voice_manifest.exists():
        archive.write(voice_manifest, arcname="generated_pack/voice/voice_manifest.json")
        
    voice_report = VOICE_ROOT / "voice_generation_report.json"
    if voice_report.exists():
        archive.write(voice_report, arcname="generated_pack/voice/voice_generation_report.json")

print("Process complete. Final zip saved to", zip_path)
